# Øvelser: Fra rå fil til document-term matrix

**Social Data Science 1: lektion 6**

I sidste uge hentede I data ned fra nettet. I dag arbejder vi med resultatet af sådan et scrape: 7.207
jobopslag fra fire portaler, indsamlet med fire forskellige scrapere i efteråret 2023.

Målet er ikke at analysere dem. Målet er at nå frem til en **datastruktur**, man kan
analysere og at kunne gøre rede for hvert eneste valg undervejs.

**Fil:** `jobapplications_fall-2023.csv`

**Sådan arbejder I:**

- Skriv hvert valg ned. De kan bruges i portfolien.
- Læs fejlbeskeder. **De første fire opgaver består næsten kun af fejlbeskeder.**
- De steder, hvor koden er givet, brug tid på at **forklare** koden, enten for jer selv---men meget gerne for hinanden. 


---

## Opgave 1: Få filen ind

**Trin 1: prøv det oplagte.** Kør cellen og læs fejlen.


In [ ]:
import pandas as pd

sti = "/Users/jeppefl/Library/CloudStorage/OneDrive-AalborgUniversitet/01_work/01_undervisning/02_sds1/03_data/jobapplications/jobapplications_fall-2023.csv"

opslag = pd.read_csv(sti)


**Trin 2: separatoren.**

Filen er lavet i et dansk regneark, hvor kommaet er decimaltegn. Derfor er
kolonnerne adskilt af noget andet.

Åbn filen i en teksteditor og kig på den første linje. Tilføj det rigtige `sep=`.


In [ ]:
# Din kode her


**Trin 3: overskriften.**

Nu kommer den ind, men kolonnerne hedder noget forkert. Kig på de første to rækker.

*Hint:* `skiprows=`


In [ ]:
# Din kode her



**Trin 4: tegnsættet.**

Print noget af teksten. Ser ord som `København` rigtig ud?

En tekstfil er bytes. Tegnsættet er aftalen om, hvilket tegn hver byte betyder, og
filen indeholder ingen oplysning om, hvilken aftale der gælder. Man gætter bare løs ... 

Prøv `encoding=` med `"utf-8"`, `"latin-1"`, `"cp1252"` og `"mac_roman"`.
Hvilken giver rigtige danske bogstaver?


In [ ]:
# Din kode her



### 1.5 Til diskussion

1. Hvad sker der, hvis I bruger det forkerte tegnsæt og ikke opdager det? Hvor i
   analysen ville det dukke op?


---

## Opgave 2: Metadata er også tekst

**Trin 1: hvad mangler?**

Kør `opslag.isna().sum()`. Kolonnen `company` mangler for de fleste opslag.

Undersøg om det er tilfældigt. Grupper efter `source` og se hvor mange der mangler
i hver kilde.


In [ ]:
print(opslag.isna().sum())
print()

mangler = opslag["company"].isna()

# Din kode her


**Trin 2: find firmanavnet.**

Det er ikke væk. Print `various_meta1` for et opslag, hvor `company` mangler.

Hvad står der, og hvordan er felterne adskilt?


In [ ]:
# Vælg et opslag hvor company mangler
uden_firma = opslag[opslag["company"].isna()]
print(uden_firma["various_meta1"].iloc[10])

**Trin 3: hent det ud.**

Del feltet op med `.split("\n")`.

Byg en ny kolonne `firma`, der indeholder den første linje.

*Hint:* `opslag["various_meta1"].str.split("\n").str[0]`


In [ ]:
# Din kode her

# Når kolonnen firma er lavet, kan denne køres:
# print(opslag[["source", "company", "firma"]].head(10))

**Trin 4: fælden.**

Tæl, hvor mange linjer der er i hvert felt:

```python
linjeantal = opslag["various_meta1"].str.split("\n").str.len()
print(linjeantal.value_counts())
```

De fleste har tre. Nogle har to. Kig på et par af dem med to linjer.

Hvad er der havnet i jeres `firma`-kolonne for netop de opslag? Og hvordan finder I
dem uden at læse alle 7.207 igennem?


In [ ]:
linjeantal = opslag["various_meta1"].str.split("\n").str.len()
print(linjeantal.value_counts())
print()

# De fleste har tre linjer. Nogle har to.
to_linjer = opslag[linjeantal == 2]
print("felter med kun to linjer:", len(to_linjer))
print()

# Din kode her


**Trin 4.1** I trin 4 fandt vi ud af, at various_meta1 ikke altid har samme struktur. Når feltet har tre linjer, er firmaet typisk linje 0. Når det kun har to linjer, er linje 0 derimod stedet.

Lad os derfor lave en ny firma-kolonne, hvor vi kun bruger various_meta1, når feltet faktisk indeholder tre linjer.

In [ ]:
# Del feltet op i linjer
linjer = opslag["various_meta1"].str.split("\n")

# Hent firmaet fra linje 0, men kun når der er tre linjer
opslag["firma"] = linjer.where(linjeantal == 3).str[0]

print(opslag["firma"].head())

opslag["firma"] = opslag["company"].fillna(opslag["firma"])

print(opslag["firma"].isna().groupby(opslag["source"]).sum())

**Trin 5: datoen.**

Tredje linje ser ca. sådan her ud: `Indrykket 24-10-2023`. Hent datoen ud og gør den til en
rigtig dato.

Hvor mange lykkes det for? Og hvilke kilder har slet ingen dato?


In [ ]:
# Din kode her
# Tredje linje ser ud som 'Indrykket 24-10-2023'
opslag["dato_raa"] = dele.str[-1]        # sidste linje, ikke [2]

dato_tekst = opslag["dato_raa"].str.replace("Indrykket ", "", regex=False)
opslag["dato"] = pd.to_datetime(dato_tekst, format="%d-%m-%Y", errors="coerce")

print("fik en dato:", opslag["dato"].notna().sum(), "af", len(opslag))
print()
print(opslag.groupby("source")["dato"].count())


**Valgfri: Fjern en specifik kilde/source** 

```
opslag["source"] != "XXXXXXXXXX"
```


In [74]:
# Din kode her

**Valgfri: Fjern observationer med NA i specifikke variable** 

```
opslag = opslag.dropna(subset=["dato", "firma", "source"])
```

In [ ]:
# Din kode her

### 2.6 Til diskussion

1. Er det bedre at rette i data eller at skrive kode, der kan håndtere alle fire
   scraperes formater?
2. Hvad skal der stå i en evt. portfolio om, at firmanavnet er hentet ud af et
   fritekstfelt for over 80% af opslagene?


---

## Opgave 3: Byg preprocessing-kæden

Hele kæden er **én pandas-Series**, som bliver "filtreret ned" uden brug af loops. 

`.explode()` folder listen af ord ud, så der er én række per ord og indekset
husker, hvilket opslag ordet kom fra.

**Trin 0: udgangspunktet.** 


In [ ]:
tekster = opslag["job_description"].dropna().astype(str)

ord = tekster.str.split().explode()

print("Tokens:", len(ord))
print("Typer: ", ord.nunique())


**Trin 1: små bogstaver.**

`.str.lower()` virker på hele kolonnen på én gang.

*Gæt først*: hvilket af de to tal (Tokens/Typer) ændrer sig?


In [ ]:
# Din kode her

print("Tokens:", len(ord))
print("Typer: ", ord.nunique())


**Trin 2: tegnsætning.**

```python
import string
tegn = string.punctuation + "«»…–—"
```

Brug `.str.strip(tegn)` på hele kolonnen. Nogle ord var kun tegnsætning og bliver
til tomme strenge, dem skal I filtrere væk med `ord[ord != ""]`.

Hvorfor bliver ord som `søger/finder` stående som ét ord?


In [ ]:
# Din kode her

print("Tokens:", len(ord))
print("Typer: ", ord.nunique())


**Trin 3: tal.**

Fjern alle ord, der indeholder et ciffer.

`.str.contains(r"\d")` giver `True` for hvert ord med et ciffer i. `~` vender
det om, så I beholder resten:

```python
ord = ord[~ord.str.contains(r"\d")]
```

Kig på ti af de ord, I lige har fjernet. Var de alle sammen støj?


In [ ]:
# Din kode her

print("Tokens:", len(ord))
print("Typer: ", ord.nunique())


**Trin 4: stopord.**


In [ ]:
stopord = set("""og i jeg det at en den til er som på de med han af for ikke
der var mig sig men et har om vi min havde ham hun nu over da fra du ud sin
dem os op man hans hvor eller hvad skal selv her alle vil blev kunne ind når
være dog jo dette dig deres end mit også under have dens hvis dine disse
hvem vores jer sådan andre nogle bliver blive kan""".split())

# Din kode her

print("stopord på listen:", len(stopord))
print("Tokens:", len(ord))
print("Typer: ", ord.nunique())

**Trin 5: sjældne ord.**

Tæl for hvert ord, i hvor mange **dokumenter** det optræder og ikke hvor mange
gange i alt.

Hintet er `.drop_duplicates()`, der gør rækkerne om til par af (dokument, ord), og
smid gengangere inden for samme dokument væk.

```python
par = ord.rename("ord").reset_index().drop_duplicates()
dok_frekvens = par["ord"].value_counts()
```

Behold derefter kun de ord, der findes i mindst fem dokumenter.


In [ ]:
par = ord.rename("ord").reset_index().drop_duplicates()
dok_frekvens = par["ord"].value_counts()

# Din kode her


print("Tokens:", len(ord))
print("Typer: ", ord.nunique())
print()
print("ord i præcis n dokument:", (dok_frekvens == 10).sum())


### 3.6 Til diskussion

1. To trin fjerner mange tokens og næsten ingen typer. To gør det modsatte. Hvilke?
2. Hvor mange ord optræder i præcis ét dokument? Hvad kan de bruges til, hvis man
   vil sammenligne dokumenter?
3. Hvor stor en del af det oprindelige vokabular er tilbage?


---

## Opgave 4: Rækkefølgen og kombinationen

Denny & Spirlings hovedpointe: syv operationer, der hver kan slås til eller fra,
giver 128 forskellige udgaver af det samme materiale.

**Trin 1: gør kæden til en funktion.**

Funktionen tager en **kolonne af tekster** ind og giver en **kolonne af ord** ud.

Fire argumenter, der kan slås til og fra.


In [84]:
def rens(tekster, lower=True, punkt=True, tal=True, stop=True):
    ord = tekster.str.split().explode()
    if lower:
        ord = ord.str.lower()
    if punkt:
        ord = ord.str.strip(tegn)
    if tal:
        ord = ord[~ord.str.contains(r"\d")]
    if stop:
        ord = ord[~ord.isin(stopord)]
    return ord[ord != ""]

print(rens(tekster.head(3)).head(8).tolist())
print(rens(tekster.head(3), lower=False, stop=False).head(8).tolist())

['søger', 'freelanceundervisere', 'levere', 'højkvalitetsundervisning', 'dansk', 'københavn', 'omegn', 'søger']
['Vi', 'søger', 'freelanceundervisere', 'der', 'kan', 'levere', 'højkvalitetsundervisning', 'i']


**Trin 2: kør alle seksten kombinationer.**

Brug de første 2.000 opslag, så det ikke tager for lang tid. Print vokabularets
størrelse for hver kombination.

Hvor stor er forskellen mellem den mindste og den største?


In [ ]:
import itertools

udsnit = tekster.head(2000)

for L, P, N, S in itertools.product([True, False], repeat=4):
    n = rens(udsnit, L, P, N, S).nunique()
    print(f"L={int(L)} P={int(P)} N={int(N)} S={int(S)}  ->  {n:>7,} unikke ord")

**Trin 3: rækkefølgen.**

Fjern stopord **før** I laver små bogstaver, i stedet for efter.

Får I det samme vokabular? Hvorfor / hvorfor ikke?


In [ ]:
# Din kode her


### 4.4 Til diskussion

1. Hvad ville I skrive i metodeafsnittet, hvis resultatet holdt på tværs af alle
   preprocessrings-tilgange? Og hvis det ikke gjorde?


---

## Opgave 5: Frekvens og hvad den ikke viser

**Trin 1: de hyppigste ord.**

`.value_counts()` tæller og sorterer i ét kald.

Print de tyve hyppigste ord i det rensede materiale.


In [ ]:
# Din kode her



**Trin 2: sproget.**

Hvor mange af opslagene er på engelsk? Et groft mål:

```python
paa_engelsk = tekster.str.lower().str.count(r"\bthe\b") >= 10
```

Hvad gør en dansk stopordsliste ved engelske stopord?

Er det et preprocessing-problem eller et korpusproblem?


In [ ]:
# Din kode her


**Trin 2.1: Fjern engelske opslag**

```
# Behold kun ikke-engelske opslag og gør preprocessringen igen
tekster = tekster.loc[~paa_engelsk]

ord = tekster.str.split().explode()

ord = ord.str.strip(tegn)
ord = ord[ord != ""]

dok_frekvens = par["ord"].value_counts()
hyppige = dok_frekvens[dok_frekvens >= 5].index
ord = ord[ord.isin(hyppige)]

ord = ord[~ord.str.fullmatch(r"\d+")]

ord = ord[~ord.isin(stopord)]

par = ord.rename("ord").reset_index().drop_duplicates()
```

Forklar hvad der sker i ovenstående kode. Er der noget at være særlig opmærksom på ved tilskrivning `tekster = ...`?

In [89]:
# Din kode her


In [ ]:
print(ord.value_counts().head(20))

**Trin 3: de domænespecifikke ord.**

Ser man bort fra de engelske, står ord som `arbejde`, `stillingen` og `opgaver`
tilbage blandt de hyppigste.

De står i næsten alle jobopslag, netop fordi det er jobopslag. Byg en
**korpusspecifik** stopordsliste ud fra dokumentfrekvensen:

```python
andel = dok_frekvens / len(tekster)
domaene = andel[andel > 0.50].index
```

Prøv flere tærskler. Over 70 % er der ingen ord tilbage. Hvad siger det?

**Fjern de domænespecifikke ord på samme måde som i fjernede stopord**.


In [ ]:
# Din kode her


**Trin 4: nøgleord.**

Sammenlign to kilder i stedet for at kigge på hele materialet under ét.

Byg en tabel med ét ord per række og en kolonne per kilde:

```python
kilde_pr_ord = ord.index.map(opslag["source"])
tabel = pd.crosstab(ord, kilde_pr_ord)
```

Regn hvert ords **andel** af tokens i hver kilde, og se på forholdet mellem de to
andele. Hvilke ord er relativt hyppigere i `job.aka.dk` end i `jobbank.dk`?

brug `.nlargest(10).round(1)` til at finde de mest hyppiste i hver kilde. 



In [ ]:
kilde_pr_ord = ord.index.map(opslag["source"])
tabel = pd.crosstab(ord, kilde_pr_ord)

# Din kode her



---

## Opgave 6: Document-term matrix

**Trin 1: byg den.**

Én række per opslag, én kolonne per ord, tællinger i cellerne.

Fordi indekset stadig husker dokumentet, er det én linje:

```python
dtm = pd.crosstab(ord.index, ord)
```
**Beskriv DTM objektet**


In [ ]:
# Din kode her
dtm = pd.crosstab(ord.index, ord)


**Trin 2: sparsity.**

Hvor stor en andel af cellerne er nul?

*Hint:* `(dtm == 0).to_numpy().mean()`


In [ ]:
# Din kode her


**Trin 3: hvad det betyder.**

Vælg to tilfældige opslag. Hvor mange ord har de til fælles?

*Hint:* `dtm.loc[i] > 0` giver en boolean per ord. `&` mellem to af dem giver de
fælles, og `.sum()` tæller dem.

Gør det ti gange. Hvad er typisk?


In [ ]:
# Din kode her



### 6.4 Til diskussion (her skal i gætte intuitivt... vi introducere det mere til lektion 7)

1. Hvert opslag er et punkt i et rum med tusindvis af dimensioner. Hvad betyder
   "afstand" mellem to punkter der?
2. Hvis næsten alle par af opslag har nul fælles ord, hvad sker der så med en
   klyngeanalyse?
3. Hvad tror I PCA gør ved det problem? (Vi tager det næste gang.)


---

## Opgave 7: spaCy

Denne del skal køres i UCloud eller lokalt.

```bash
%pip install spacy
```

```python
import sys

!{sys.executable} -m spacy download da_core_news_sm
```

**Trin 1: kør pipelinen på én tekst.**


In [ ]:
%pip install spacy

In [ ]:
import sys

!{sys.executable} -m spacy download da_core_news_sm

In [ ]:
import spacy

nlp = spacy.load("da_core_news_sm")
doc = nlp("Vi søger en erfaren socialrådgiver til vores team i Aarhus.")

for tok in doc:
    print(tok.text, tok.lemma_, tok.pos_, tok.is_stop)


**Trin 2: stemming mod lemmatisering.**

Kør pipelinen på ordene `søger`, `søgte`, `søgt`, `universitet`, `universel`.

```
for ordet in ord:
    doc = nlp(ordet)
    print(ordet, "->", doc[0].lemma_)
```

Hvad bliver de til? Hvad ville en simpel regel (fx. klip de sidste to bogstaver af)
have gjort ved de samme ord?


In [ ]:
# Din kode her



**Trin 3: ordklasser som filter.**

Kør pipelinen på 200 opslag og behold kun navneord og tillægsord.

Sammenlign `lemmaer.nunique()` med det, I fik af regelversionen på de samme 200
opslag. Hvor stor er forskellen?


In [ ]:
# Din kode her
# 200 opslag gennem pipelinen
udsnit200 = tekster.head(200)

lemmaer = []

for doc in nlp.pipe(udsnit200):

    for token in doc:

        if token.is_stop:
            continue

        if token.is_punct:
            continue

        if token.like_num:
            continue

        if token.pos_ not in ("NOUN", "PROPN", "ADJ"):
            continue

        lemmaer.append(token.lemma_.lower())

lemmaer = pd.Series(lemmaer)

# Sammenlign med regelbaseret rensning
regel = #SKRIV EGEN KODE HER

print("regler:", len(regel), "tokens |", regel.nunique(), "typer")
print("spaCy: ", len(lemmaer), "tokens |", lemmaer.nunique(), "typer")
print()
print("spaCy beholder", round(len(lemmaer)/len(regel)*100), "% af tokens")


**Trin 4: se efter for fejl.**

Modellen er trænet på dansk nyhedstekst. Jobopslag er ikke nyhedstekst, og en del af
materialet er på engelsk.

Find mindst tre steder, hvor lemmatiseringen eller ordklassen er forkert. Hvilken
slags ord går det galt på?

**Vælg en tekst ved at ændre `iloc[N]`**


In [ ]:
# Din kode her
doc = nlp(tekster.iloc[10][:400])

for token in doc:
    if not token.is_punct and not token.is_space:
        print(token.text, "->", token.lemma_, "(", token.pos_, ")")


**Trin 5: En færdig SpaCy pipeline, der kan danne grundlag for analyse**

In [101]:
def preprocesser(tekst):
    doc = nlp(tekst)

    ord = []

    for token in doc:
        if token.is_stop:
            continue
        if token.is_punct:
            continue
        if token.like_num:
            continue
        if token.pos_ not in ("NOUN", "PROPN", "ADJ"):
            continue

        ord.append(token.lemma_.lower())

    return " ".join(ord)

In [ ]:
print(opsalg["job_description"].isna().sum())

**OBS BLOKKEN NEDENFOR TOG CA. 10 MIN. AT KØRE PÅ MIN COMPUTER, LOKALT.**

In [103]:
data = opslag[["source", "firma", "job_description"]].copy()
data = data.dropna(subset=["job_description"])
data["clean_text"] = data["job_description"].apply(preprocesser)

### 7.5 Til diskussion

1. Reglerne kørte på sekunder. spaCy tager minutter. Hvornår er det pengene værd?
2. I kan læse jeres egen stopordsliste. I kan ikke læse en model. Betyder det noget
   for, hvad I kan skrive i metodeafsnittet?


---

## Opgave 8: Til portfolien

Skriv en preprocessing-protokol for jeres eget materiale.

1. **Hvert valg, i rækkefølge.** Hvad gjorde I, og hvorfor netop det?

2. **Hvad kostede hvert valg?** Ét konkret eksempel per trin på noget, der gik tabt.

3. **Tokens og typer** før og efter. Hvor stor blev reduktionen?

4. **Robusthed.** Prøvede I flere kombinationer? Holdt resultatet?

Protokollen skal være detaljeret nok til, at en anden kan gentage den præcist.
Det er hele pointen i Denny & Spirlings artikel.


In [ ]:
# Din kode her

